# Enrich Silver+Gold P1/P2 + age/geo (v2)

Grain: account → campaign → adset → ad → date → metrics



**Incremental:** default `FULL_REFRESH=False` merges new dates; existing Silver/Gold history is kept. Large backfills are blocked unless `FULL_REFRESH=True`.


In [ ]:
WORKSPACE_ID = "718e8176-5d40-4a9c-88ff-50ac97ac49ba"
LAKEHOUSE_ID = "981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
BASE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
META_BRONZE = f"{BASE}/Files/Development/Bronze/Meta_ads"
META_BRONZE_PROD = f"{BASE}/Files/Bronze"
GOOGLE_SILVER = f"{BASE}/Files/Development/Silver/GoogleAds"
META_SILVER = f"{BASE}/Files/Silver/meta_ads"
GOLD_FILES_META = f"{BASE}/Files/Gold"
GOLD_FILES_GOOGLE = f"{BASE}/Files/Development/Gold/GoogleAds"
GOLD_FILES_UNIFIED = f"{BASE}/Files/Development/Gold"
S = "Gold"
print("ready")

FULL_REFRESH = False
INCREMENTAL_LOOKBACK_DAYS = 2
INCREMENTAL_MAX_DAYS = 14


In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable
import json

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")

# Incremental helpers
from delta.tables import DeltaTable

def _table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(1).collect(); return True
    except Exception:
        return False

def _path_is_delta(path: str) -> bool:
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except Exception:
        return False

def _max_date(table_or_path: str, date_col: str, is_path: bool = False):
    try:
        df = spark.read.format("delta").load(table_or_path) if is_path else spark.table(table_or_path)
        return df.agg(F.max(F.col(date_col)).alias("m")).collect()[0]["m"]
    except Exception:
        return None

def filter_by_watermark(df, date_col: str, target: str, is_path: bool = False):
    if FULL_REFRESH:
        print(f"[FULL_REFRESH] no watermark filter: {target}"); return df
    exists = _path_is_delta(target) if is_path else _table_exists(target)
    if not exists:
        print(f"[INCR] target missing → first load: {target}"); return df
    wm = _max_date(target, date_col, is_path=is_path)
    if wm is None:
        print(f"[INCR] empty watermark → full batch: {target}"); return df
    out = df.filter(F.col(date_col).isNotNull() & (F.col(date_col) >= F.date_sub(F.lit(wm), int(INCREMENTAL_LOOKBACK_DAYS))))
    st = out.agg(F.min(date_col).alias("mn"), F.max(date_col).alias("mx"), F.count(F.lit(1)).alias("n")).collect()[0]
    print(f"[INCR] {target} wm={wm} lookback={INCREMENTAL_LOOKBACK_DAYS}d rows={st['n']} range={st['mn']}..{st['mx']}")
    if st["n"] and st["mn"] is not None and st["mx"] is not None and (st["mx"] - st["mn"]).days > int(INCREMENTAL_MAX_DAYS):
        raise ValueError(f"Incremental batch for {target} spans {(st['mx']-st['mn']).days} days (> {INCREMENTAL_MAX_DAYS}). Refusing large backfill.")
    return out

def merge_or_overwrite_table(df, target: str, keys, partition_cols=None, stamp_col="gold_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _table_exists(target):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if partition_cols: w = w.partitionBy(*partition_cols)
        w.saveAsTable(target); mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()); mode = "MERGE"
    print(f"[OK] {target} ({mode}) total={spark.table(target).count():,}")

def merge_or_overwrite_path(df, path: str, keys, partition_cols=None, stamp_col="_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _path_is_delta(path):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("mergeSchema", "true")
        if partition_cols: w = w.partitionBy(*partition_cols)
        w.save(path); mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forPath(spark, path).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()); mode = "MERGE"
    print(f"[OK] {path} ({mode}) total={spark.read.format('delta').load(path).count():,}")


def write_delta(df, path=None, table=None, keys=None, partition_cols=None, date_col=None):
    if not keys:
        raise ValueError("keys required for incremental write_delta")
    batch = df
    target = f"{S}.{table}" if table else path
    is_path = table is None
    if date_col and target:
        batch = filter_by_watermark(batch, date_col, target, is_path=is_path)
        if len(batch.take(1)) == 0:
            print(f"[SKIP] {target}: no new incremental rows"); return
    if table:
        merge_or_overwrite_table(batch, f"{S}.{table}", keys, partition_cols=partition_cols, stamp_col="gold_processed_at")
    if path:
        merge_or_overwrite_path(batch, path, keys, partition_cols=partition_cols, stamp_col="gold_processed_at")

def read_csv(path):
    return spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)

def load_bronze(name):
    try:
        df = read_csv(f"{META_BRONZE}/{name}")
        n = df.count()
        if n == 0:
            raise ValueError("empty")
        print(f"[BRONZE] {META_BRONZE}/{name}: {n}")
        return df
    except Exception as e:
        print(f"[FALLBACK] {name}: {e}")
        df = read_csv(f"{META_BRONZE_PROD}/{name}")
        print(f"[BRONZE] prod {name}: {df.count()}")
        return df

# Parse helpers via Python UDF (robust on Fabric)
@F.udf(T.MapType(T.StringType(), T.DoubleType()))
def parse_actions_map(raw):
    out = {
        "lead": 0.0, "link_click": 0.0, "landing_page_view": 0.0,
        "post_engagement": 0.0, "video_view": 0.0,
    }
    try:
        obj = json.loads(raw) if raw else {}
        for a in obj.get("actions") or []:
            if not isinstance(a, dict):
                continue
            t = a.get("action_type")
            if t in out:
                try:
                    out[t] += float(a.get("value") or 0)
                except Exception:
                    pass
    except Exception:
        pass
    return out

@F.udf(T.StructType([
    T.StructField("age_min", T.IntegerType()),
    T.StructField("age_max", T.IntegerType()),
    T.StructField("age_range", T.StringType()),
    T.StructField("geo_country", T.StringType()),
    T.StructField("geo_regions", T.StringType()),
    T.StructField("geo_cities", T.StringType()),
]))
def parse_targeting(raw):
    age_min = age_max = None
    age_range = geo_country = geo_regions = geo_cities = None
    try:
        obj = json.loads(raw) if raw else {}
        t = obj.get("targeting") or {}
        age_min = t.get("age_min")
        age_max = t.get("age_max")
        ar = t.get("age_range")
        if isinstance(ar, list) and len(ar) >= 2:
            age_range = f"{ar[0]}-{ar[1]}"
        elif age_min is not None and age_max is not None:
            age_range = f"{age_min}-{age_max}"
        geo = t.get("geo_locations") or {}
        countries = geo.get("countries") or []
        cities = geo.get("cities") or []
        if not countries:
            countries = sorted({c.get("country") for c in cities if isinstance(c, dict) and c.get("country")})
        regions = sorted({c.get("region") for c in cities if isinstance(c, dict) and c.get("region")})
        for r in geo.get("regions") or []:
            if isinstance(r, dict) and r.get("name"):
                regions.append(r["name"])
        regions = sorted(set(regions))
        city_names = [c.get("name") for c in cities if isinstance(c, dict) and c.get("name")]
        geo_country = ", ".join(countries) if countries else None
        geo_regions = ", ".join(regions) if regions else None
        geo_cities = ", ".join(city_names) if city_names else None
        if age_min is not None:
            age_min = int(age_min)
        if age_max is not None:
            age_max = int(age_max)
    except Exception:
        pass
    return (age_min, age_max, age_range, geo_country, geo_regions, geo_cities)

print("udfs ready")



In [ ]:
# SILVER insights
b = load_bronze("meta_ad_insights")
j = F.from_json(F.col("raw_json"), "map<string,string>")
# also keep typed parse via get_json_object for reliability
am = parse_actions_map(F.col("raw_json"))
silver_insights = (
    b.select(
        F.col("connector_id"), F.col("tenant_id"), F.col("account_id"), F.col("account_name"),
        F.col("platform"), F.col("batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        F.to_date("extraction_start_date").alias("extraction_start_date"),
        F.to_date("extraction_end_date").alias("extraction_end_date"),
        F.get_json_object("raw_json", "$.ad_id").alias("ad_id"),
        F.get_json_object("raw_json", "$.adset_id").alias("adset_id"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.ad_name").alias("ad_name"),
        F.to_date(F.get_json_object("raw_json", "$.date_start")).alias("date_start"),
        F.to_date(F.get_json_object("raw_json", "$.date_stop")).alias("date_stop"),
        F.get_json_object("raw_json", "$.impressions").cast("long").alias("impressions"),
        F.get_json_object("raw_json", "$.clicks").cast("long").alias("clicks"),
        F.get_json_object("raw_json", "$.unique_clicks").cast("long").alias("unique_clicks"),
        F.get_json_object("raw_json", "$.inline_link_clicks").cast("long").alias("inline_link_clicks"),
        F.get_json_object("raw_json", "$.reach").cast("long").alias("reach"),
        F.get_json_object("raw_json", "$.spend").cast("double").alias("spend"),
        F.get_json_object("raw_json", "$.frequency").cast("double").alias("frequency"),
        F.get_json_object("raw_json", "$.ctr").cast("double").alias("ctr"),
        F.get_json_object("raw_json", "$.unique_ctr").cast("double").alias("unique_ctr"),
        F.get_json_object("raw_json", "$.cpc").cast("double").alias("cpc"),
        F.get_json_object("raw_json", "$.cpm").cast("double").alias("cpm"),
        F.get_json_object("raw_json", "$.cpp").cast("double").alias("cpp"),
        am["lead"].alias("leads"),
        am["link_click"].alias("link_clicks"),
        am["landing_page_view"].alias("landing_page_views"),
        am["post_engagement"].alias("post_engagement"),
        am["video_view"].alias("video_views_3s"),
    )
    .withColumn(
        "cost_per_lead",
        F.when(F.col("leads") > 0, F.col("spend") / F.col("leads")).otherwise(F.lit(None).cast("double")),
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
    .dropDuplicates(["tenant_id", "ad_id", "date_start"])
)
print("insights", silver_insights.count())
silver_insights.agg(F.sum("leads"), F.sum("link_clicks"), F.sum("landing_page_views"), F.sum("post_engagement"), F.sum("video_views_3s")).show()
write_delta(silver_insights, path=f"{META_SILVER}/silver_meta_ad_insights", keys=["ad_id","date_start"], date_col="date_start")
write_delta(silver_insights, path=f"{BASE}/Files/Development/Silver/meta_ads/silver_meta_ad_insights", keys=["ad_id","date_start"], date_col="date_start")



In [ ]:
# SILVER adsets + age/geo
b = load_bronze("meta_adsets")
tg = parse_targeting(F.col("raw_json"))
# budgets: Meta API cents -> major currency if values look like cents
raw_daily = F.get_json_object("raw_json", "$.daily_budget").cast("double")
raw_life = F.get_json_object("raw_json", "$.lifetime_budget").cast("double")
silver_adsets = (
    b.select(
        F.col("tenant_id"), F.col("account_id"), F.col("account_name"),
        F.coalesce(F.col("platform"), F.lit("meta")).alias("platform"),
        F.col("batch_id"), F.to_timestamp("ingestion_time").alias("ingestion_time"),
        F.to_date("extraction_start_date").alias("extraction_start_date"),
        F.to_date("extraction_end_date").alias("extraction_end_date"),
        F.get_json_object("raw_json", "$.id").alias("adset_id"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.name").alias("adset_name"),
        F.get_json_object("raw_json", "$.status").alias("status"),
        F.get_json_object("raw_json", "$.optimization_goal").alias("optimization_goal"),
        F.get_json_object("raw_json", "$.billing_event").alias("billing_event"),
        F.get_json_object("raw_json", "$.bid_strategy").alias("bid_strategy"),
        (raw_daily / F.lit(100.0)).alias("daily_budget"),
        (raw_life / F.lit(100.0)).alias("lifetime_budget"),
        F.to_timestamp(F.get_json_object("raw_json", "$.created_time")).alias("created_time"),
        F.to_timestamp(F.get_json_object("raw_json", "$.updated_time")).alias("updated_time"),
        tg["age_min"].alias("age_min"),
        tg["age_max"].alias("age_max"),
        tg["age_range"].alias("age_range"),
        tg["geo_country"].alias("geo_country"),
        tg["geo_regions"].alias("geo_regions"),
        tg["geo_cities"].alias("geo_cities"),
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
    .dropDuplicates(["tenant_id", "adset_id"])
)
print("adsets", silver_adsets.count())
silver_adsets.select("adset_name", "age_range", "geo_country", "geo_regions", "geo_cities", "optimization_goal", "billing_event").show(5, truncate=40)
write_delta(silver_adsets, path=f"{META_SILVER}/silver_meta_adsets", keys=["adset_id"])
write_delta(silver_adsets, path=f"{BASE}/Files/Development/Silver/meta_ads/silver_meta_adsets", keys=["adset_id"])



In [ ]:
# SILVER campaigns + ads
b = load_bronze("meta_campaigns")
raw_daily = F.get_json_object("raw_json", "$.daily_budget").cast("double")
raw_rem = F.get_json_object("raw_json", "$.budget_remaining").cast("double")
silver_campaigns = (
    b.select(
        F.col("tenant_id"), F.col("account_id"), F.col("account_name"),
        F.coalesce(F.col("platform"), F.lit("meta")).alias("platform"),
        F.col("batch_id"), F.to_timestamp("ingestion_time").alias("ingestion_time"),
        F.to_date("extraction_start_date").alias("extraction_start_date"),
        F.to_date("extraction_end_date").alias("extraction_end_date"),
        F.get_json_object("raw_json", "$.id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.name").alias("campaign_name"),
        F.get_json_object("raw_json", "$.status").alias("status"),
        F.get_json_object("raw_json", "$.objective").alias("objective"),
        F.get_json_object("raw_json", "$.buying_type").alias("buying_type"),
        F.get_json_object("raw_json", "$.bid_strategy").alias("bid_strategy"),
        (raw_daily / F.lit(100.0)).alias("daily_budget"),
        (raw_rem / F.lit(100.0)).alias("budget_remaining"),
        F.to_timestamp(F.get_json_object("raw_json", "$.created_time")).alias("created_time"),
        F.to_timestamp(F.get_json_object("raw_json", "$.updated_time")).alias("updated_time"),
        F.to_timestamp(F.get_json_object("raw_json", "$.start_time")).alias("start_time"),
        F.to_timestamp(F.get_json_object("raw_json", "$.stop_time")).alias("stop_time"),
    ).withColumn("_silver_processed_at", F.current_timestamp()).dropDuplicates(["tenant_id", "campaign_id"])
)

b = load_bronze("meta_ads")
silver_ads = (
    b.select(
        F.col("tenant_id"), F.col("account_id"), F.col("account_name"),
        F.coalesce(F.col("platform"), F.lit("meta")).alias("platform"),
        F.col("batch_id"), F.to_timestamp("ingestion_time").alias("ingestion_time"),
        F.to_date("extraction_start_date").alias("extraction_start_date"),
        F.to_date("extraction_end_date").alias("extraction_end_date"),
        F.get_json_object("raw_json", "$.id").alias("ad_id"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.adset_id").alias("adset_id"),
        F.get_json_object("raw_json", "$.name").alias("ad_name"),
        F.get_json_object("raw_json", "$.status").alias("status"),
        F.get_json_object("raw_json", "$.effective_status").alias("effective_status"),
        F.get_json_object("raw_json", "$.creative.id").alias("creative_id"),
        F.get_json_object("raw_json", "$.creative.thumbnail_url").alias("thumbnail_url"),
        F.get_json_object("raw_json", "$.creative.title").alias("ad_title"),
        F.get_json_object("raw_json", "$.creative.body").alias("ad_body"),
        F.to_timestamp(F.get_json_object("raw_json", "$.created_time")).alias("created_time"),
        F.to_timestamp(F.get_json_object("raw_json", "$.updated_time")).alias("updated_time"),
    ).withColumn("_silver_processed_at", F.current_timestamp()).dropDuplicates(["tenant_id", "ad_id"])
)
print("campaigns", silver_campaigns.count(), "ads", silver_ads.count())
write_delta(silver_campaigns, path=f"{META_SILVER}/silver_meta_campaigns", keys=["campaign_id"])
write_delta(silver_ads, path=f"{META_SILVER}/silver_meta_ads", keys=["ad_id"])
write_delta(silver_campaigns, path=f"{BASE}/Files/Development/Silver/meta_ads/silver_meta_campaigns", keys=["campaign_id"])
write_delta(silver_ads, path=f"{BASE}/Files/Development/Silver/meta_ads/silver_meta_ads", keys=["ad_id"])
print("SILVER_DONE")



In [ ]:
# GOLD META
campaigns = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_campaigns")
adsets = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_adsets")
ads = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_ads")
insights = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_ad_insights")

rpt_meta = (
    insights.alias("i")
    .withColumn("full_date", F.to_date("i.date_start"))
    .join(campaigns.alias("c"), F.col("i.campaign_id") == F.col("c.campaign_id"), "left")
    .join(adsets.alias("s"), F.col("i.adset_id") == F.col("s.adset_id"), "left")
    .join(ads.alias("ad"), F.col("i.ad_id") == F.col("ad.ad_id"), "left")
    .where(F.col("full_date").isNotNull())
    .select(
        F.col("full_date"), F.year("full_date").alias("year"), F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"), F.date_format("full_date", "EEEE").alias("day_name"),
        F.col("i.tenant_id"), F.col("i.connector_id"), F.col("i.account_id"), F.col("i.account_name"),
        F.lit("meta").alias("platform"),
        F.col("i.campaign_id"), F.col("c.campaign_name"), F.col("c.objective").alias("campaign_objective"),
        F.col("c.status").alias("campaign_status"), F.col("c.buying_type"),
        F.col("c.bid_strategy").alias("campaign_bid_strategy"),
        F.col("c.daily_budget").alias("daily_budget_inr"), F.col("c.daily_budget").alias("campaign_daily_budget_inr"),
        F.col("c.budget_remaining"),
        F.col("i.adset_id"), F.col("s.adset_name"), F.col("s.status").alias("adset_status"),
        F.col("s.optimization_goal"), F.col("s.billing_event"), F.col("s.bid_strategy").alias("adset_bid_strategy"),
        F.col("s.age_min"), F.col("s.age_max"), F.col("s.age_range"),
        F.col("s.geo_country"), F.col("s.geo_regions"), F.col("s.geo_cities"),
        F.col("i.ad_id"), F.coalesce(F.col("ad.ad_name"), F.col("i.ad_name")).alias("ad_name"),
        F.col("ad.status").alias("ad_status"), F.col("ad.effective_status").alias("ad_effective_status"),
        F.col("ad.creative_id"), F.col("ad.ad_title"), F.col("ad.ad_body"),
        F.col("i.impressions").cast("double"), F.col("i.reach").cast("double"), F.col("i.frequency").cast("double"),
        F.col("i.clicks").cast("double"), F.col("i.unique_clicks").cast("double"),
        F.col("i.inline_link_clicks").cast("double"), F.col("i.unique_ctr").cast("double"),
        F.col("i.spend").cast("double").alias("spend"), F.col("i.spend").cast("double").alias("spend_inr"),
        F.col("i.cpc").cast("double"), F.col("i.cpm").cast("double"), F.col("i.cpp").cast("double"), F.col("i.ctr").cast("double"),
        F.col("i.leads").cast("double"), F.col("i.cost_per_lead").cast("double"),
        F.col("i.link_clicks").cast("double"), F.col("i.landing_page_views").cast("double"),
        F.col("i.post_engagement").cast("double"), F.col("i.video_views_3s").cast("double"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
print("meta gold", rpt_meta.count())
rpt_meta.select("account_name","campaign_name","adset_name","ad_name","full_date","spend","clicks","leads","age_range","geo_cities").show(5, truncate=35)
write_delta(rpt_meta, table="rpt_meta_ad_performance_daily", path=f"{GOLD_FILES_META}/rpt_meta_ad_performance_daily", keys=["account_id","campaign_id","adset_id","ad_id","full_date"], partition_cols=["full_date"], date_col="full_date")



In [ ]:
# GOLD GOOGLE
g_campaigns = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_campaigns")
g_adgroups = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_adgroups")
g_ads = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_ads")
g_perf = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_ad_performance")
for c in ["connector_id", "tenant_id"]:
    if c not in g_perf.columns:
        g_perf = g_perf.withColumn(c, F.lit(None).cast("string"))

g_camp = g_campaigns.select("campaign_id","campaign_name","status","channel_type","daily_budget_inr").dropDuplicates(["campaign_id"])
g_ag = g_adgroups.select("adgroup_id","adgroup_name","status").dropDuplicates(["adgroup_id"])
g_ad = g_ads.select("ad_id","ad_type","status","final_urls","headline","description").dropDuplicates(["ad_id"])

rpt_google = (
    g_perf.alias("p")
    .join(g_camp.alias("c"), "campaign_id", "left")
    .join(g_ag.alias("g"), F.col("p.adgroup_id") == F.col("g.adgroup_id"), "left")
    .join(g_ad.alias("ad"), "ad_id", "left")
    .select(
        F.coalesce(F.col("p.platform"), F.lit("google_ads")).alias("platform"),
        F.to_date(F.col("p.date")).alias("full_date"),
        F.year(F.to_date("p.date")).alias("year"), F.month(F.to_date("p.date")).alias("month"),
        F.date_format(F.to_date("p.date"), "MMMM").alias("month_name"),
        F.date_format(F.to_date("p.date"), "EEEE").alias("day_name"),
        F.col("p.tenant_id"), F.col("p.connector_id"), F.col("p.account_id"), F.col("p.account_name"),
        F.col("p.campaign_id"), F.col("c.campaign_name"), F.col("c.status").alias("campaign_status"),
        F.col("c.channel_type").alias("campaign_channel_or_objective"),
        F.col("c.daily_budget_inr").alias("daily_budget_inr"), F.col("c.daily_budget_inr").alias("campaign_daily_budget_inr"),
        F.lit(None).cast("string").alias("buying_type"), F.lit(None).cast("string").alias("campaign_bid_strategy"),
        F.lit(None).cast("double").alias("budget_remaining"),
        F.col("p.adgroup_id").alias("adset_id"), F.col("p.adgroup_id").alias("adset_or_adgroup_id"),
        F.col("g.adgroup_name").alias("adset_name"), F.col("g.adgroup_name").alias("adset_or_adgroup_name"),
        F.col("g.status").alias("adset_status"), F.col("g.status").alias("adset_or_adgroup_status"),
        F.lit(None).cast("string").alias("optimization_goal"), F.lit(None).cast("string").alias("billing_event"),
        F.lit(None).cast("string").alias("adset_bid_strategy"),
        F.lit(None).cast("int").alias("age_min"), F.lit(None).cast("int").alias("age_max"),
        F.lit(None).cast("string").alias("age_range"), F.lit(None).cast("string").alias("geo_country"),
        F.lit(None).cast("string").alias("geo_regions"), F.lit(None).cast("string").alias("geo_cities"),
        F.col("p.ad_id"), F.col("ad.headline").alias("ad_name"), F.col("ad.ad_type"), F.col("ad.status").alias("ad_status"),
        F.col("ad.final_urls"), F.lit(None).cast("string").alias("creative_id"),
        F.col("ad.headline").alias("ad_title"), F.col("ad.description").alias("ad_body"),
        F.col("ad.headline").alias("headline"), F.col("ad.description").alias("description"),
        F.col("p.impressions").cast("double"), F.lit(None).cast("double").alias("reach"), F.lit(None).cast("double").alias("frequency"),
        F.col("p.clicks").cast("double"), F.lit(None).cast("double").alias("unique_clicks"),
        F.lit(None).cast("double").alias("inline_link_clicks"), F.lit(None).cast("double").alias("unique_ctr"),
        F.col("p.ctr").cast("double"), F.col("p.spend_inr").cast("double").alias("spend"), F.col("p.spend_inr").cast("double").alias("spend_inr"),
        F.col("p.average_cpc").cast("double").alias("cpc"),
        F.when(F.col("p.impressions") > 0, (F.col("p.spend_inr")/F.col("p.impressions"))*1000).otherwise(F.lit(None)).alias("cpm"),
        F.lit(None).cast("double").alias("cpp"),
        F.col("p.conversions").cast("double").alias("conversions"), F.col("p.conversions").cast("double").alias("leads"),
        F.col("p.conversions_value").cast("double"), F.col("p.cost_per_conversion").cast("double"),
        F.col("p.cost_per_conversion").cast("double").alias("cost_per_lead"), F.col("p.roas").cast("double"),
        F.lit(None).cast("double").alias("link_clicks"), F.lit(None).cast("double").alias("landing_page_views"),
        F.lit(None).cast("double").alias("post_engagement"), F.lit(None).cast("double").alias("video_views_3s"),
        F.lit(None).cast("double").alias("engagements"), F.lit(None).cast("double").alias("video_views"),
        F.current_timestamp().alias("gold_processed_at"),
    ).where(F.col("full_date").isNotNull())
)
print("google gold", rpt_google.count())
write_delta(rpt_google, table="rpt_google_ad_performance_daily", path=f"{GOLD_FILES_GOOGLE}/rpt_google_ad_performance_daily", keys=["account_id","campaign_id","adset_id","ad_id","full_date"], partition_cols=["full_date"], date_col="full_date")



In [ ]:
# UNIFIED
meta_u = spark.table(f"{S}.rpt_meta_ad_performance_daily").select(
    F.lit("meta").alias("platform"), "full_date","year","month","month_name","day_name",
    "tenant_id","connector_id","account_id","account_name",
    "campaign_id","campaign_name","campaign_status",
    F.col("campaign_objective").alias("campaign_channel_or_objective"),
    "daily_budget_inr","campaign_daily_budget_inr","buying_type","campaign_bid_strategy","budget_remaining",
    "adset_id", F.col("adset_id").alias("adset_or_adgroup_id"),
    F.col("adset_name"), F.col("adset_name").alias("adset_or_adgroup_name"),
    F.col("adset_status"), F.col("adset_status").alias("adset_or_adgroup_status"),
    "optimization_goal","billing_event","adset_bid_strategy",
    "age_min","age_max","age_range","geo_country","geo_regions","geo_cities",
    "ad_id","ad_name", F.lit(None).cast("string").alias("ad_type"), "ad_status",
    F.lit(None).cast("string").alias("final_urls"), "creative_id",
    F.col("ad_title").alias("headline"), F.col("ad_body").alias("description"), "ad_title","ad_body",
    "impressions","reach","frequency","clicks","unique_clicks","inline_link_clicks","unique_ctr",
    "ctr","cpm","cpp","cpc","spend","spend_inr","leads","cost_per_lead",
    "link_clicks","landing_page_views","post_engagement","video_views_3s",
    F.lit(None).cast("double").alias("conversions"), F.lit(None).cast("double").alias("conversions_value"),
    F.lit(None).cast("double").alias("cost_per_conversion"), F.lit(None).cast("double").alias("roas"),
    F.lit(None).cast("double").alias("engagements"), F.lit(None).cast("double").alias("video_views"),
    "gold_processed_at",
)
google_u = spark.table(f"{S}.rpt_google_ad_performance_daily").select(
    "platform","full_date","year","month","month_name","day_name",
    "tenant_id","connector_id","account_id","account_name",
    "campaign_id","campaign_name","campaign_status","campaign_channel_or_objective",
    "daily_budget_inr","campaign_daily_budget_inr","buying_type","campaign_bid_strategy","budget_remaining",
    "adset_id","adset_or_adgroup_id","adset_name","adset_or_adgroup_name","adset_status","adset_or_adgroup_status",
    "optimization_goal","billing_event","adset_bid_strategy",
    "age_min","age_max","age_range","geo_country","geo_regions","geo_cities",
    "ad_id","ad_name","ad_type","ad_status","final_urls","creative_id","headline","description","ad_title","ad_body",
    "impressions","reach","frequency","clicks","unique_clicks","inline_link_clicks","unique_ctr",
    "ctr","cpm","cpp","cpc","spend","spend_inr","leads","cost_per_lead",
    "link_clicks","landing_page_views","post_engagement","video_views_3s",
    "conversions","conversions_value","cost_per_conversion","roas","engagements","video_views",
    "gold_processed_at",
)
unified = meta_u.unionByName(google_u)
print("unified", unified.count())
unified.groupBy("platform").agg(F.count("*").alias("rows"), F.round(F.sum("spend"),2).alias("spend"), F.round(F.sum("leads"),2).alias("leads")).show()
unified.select(
    "account_name","campaign_name","adset_name","ad_name","full_date","spend","clicks","leads","cost_per_lead","age_range","geo_cities"
).where(F.col("platform")=="meta").show(8, truncate=35)
write_delta(unified, table="rpt_unified_ad_performance", path=f"{GOLD_FILES_UNIFIED}/rpt_unified_ad_performance", keys=["platform","account_id","campaign_id","adset_id","ad_id","full_date"], partition_cols=["platform","full_date"], date_col="full_date")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_unified_ad_performance AS SELECT * FROM {S}.rpt_unified_ad_performance")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_meta_ad_performance AS SELECT * FROM {S}.rpt_meta_ad_performance_daily")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_google_ad_performance AS SELECT * FROM {S}.rpt_google_ad_performance_daily")
req = ["leads","cost_per_lead","reach","inline_link_clicks","unique_ctr","link_clicks","landing_page_views",
       "optimization_goal","creative_id","billing_event","budget_remaining","post_engagement","video_views_3s",
       "age_min","age_max","age_range","geo_country","geo_regions","geo_cities","headline","description"]
cols = set(spark.table(f"{S}.rpt_unified_ad_performance").columns)
for c in req:
    print(c, "OK" if c in cols else "MISSING")
print("ENRICH_PRIORITY_AGE_GEO_COMPLETE")

